In [1]:
#대서산업 TeamViewer로 부터 이미지 폴더를 복사하여 붙여넣기 한 후, 폴더 내 파일명을 리스트업하는 코드
from pathlib import Path

folder_path = Path("../dataset")
src_folder = folder_path / "20260429"
files = [f for f in src_folder.iterdir() if f.is_file() and 'front' in f.name]

print(len(files))

109


In [2]:
#대서산업 이미지에서 front가 포함된 파일명만 리스트업하여 dataset 폴더(roboflow 업로드용)로 복사하는 코드
import shutil
import os

# 1. 경로 설정
dest_folder = folder_path / 'tmp'
dest_folder.mkdir(parents=True, exist_ok=True)

# 기존 파일 삭제 로직
for item in dest_folder.iterdir():
    if item.is_file():
        item.unlink()  # 파일 삭제
    elif item.is_dir():
        shutil.rmtree(item)  # 하위 폴더 삭제

print("기존 dataset 폴더 내용 삭제 완료")

# 2. 파일 복사 작업
n_files = 0
for file_path in files:
    target_path = dest_folder / file_path.name
    shutil.copy2(file_path, target_path)
    n_files += 1

print("--- 작업 완료 ---", n_files)

기존 dataset 폴더 내용 삭제 완료
--- 작업 완료 --- 109


In [4]:
#대서산업 이미지를 roboflow에 업로드하는 코드
import roboflow

rf = roboflow.Roboflow(api_key="3VVAbbWfxaxc1G0QuWFV")

# List all projects for your workspace
workspace = rf.workspace()

# get a specific project
project = rf.workspace().project("quarry-x-object-detection")

# list all versions in a specific project
# project.versions()

# Upload data set to a new/existing project
workspace.upload_dataset(
    str(dest_folder), # This is your dataset path
    "quarry-x-object-detection", # This will either create or get a dataset with the given ID
    num_workers=10,
    project_license="MIT",
    project_type="object-detection",
    batch_name=None,
    num_retries=0,
    is_prediction=False #optional, set to True if the dataset is not ground truth and needs approval
)

loading Roboflow workspace...
loading Roboflow workspace...
loading Roboflow project...
loading Roboflow project...
Uploading to existing project -vfeut/quarry-x-object-detection


100%|██████████| 109/109 [00:00<?, ?it/s]


[UPLOADED] ../dataset/tmp/front_1777416835603_대구06라5245.jpg (Pea4Z8ERjYPHm6fEawwj) [2.7s]
[UPLOADED] ../dataset/tmp/front_1777419827072_대구06라5245.jpg (cYFQHDOU6xsqEcZ7csUW) [3.0s]
[UPLOADED] ../dataset/tmp/front_1777416834031_대구06라5245.jpg (ANdgdU7OEc576jAcMUEH) [3.0s]
[UPLOADED] ../dataset/tmp/front_1777419725549_006마8899.jpg (KSI51ORLVsbJQhxuyrIX) [3.0s]
[UPLOADED] ../dataset/tmp/front_1777416721279_006마8899.jpg (RZyZJR7lXb7ODw3TV8RG) [3.1s]
[UPLOADED] ../dataset/tmp/front_1777420068752_경북06모6547.jpg (m22iwxkmiEyHMnzaFVnU) [2.0s]
[UPLOADED] ../dataset/tmp/front_1777420957018_006나7520.jpg (uFyzYaEVFm7raLcpCWKd) [2.1s]
[UPLOADED] ../dataset/tmp/front_1777421066083_대구06라5226.jpg (5KKyEOvx8lFrOCyiwOzE) [1.9s]
[DUPLICATE] ../dataset/tmp/front_1777421067599_대구06라5226.jpg (5KKyEOvx8lFrOCyiwOzE) [2.0s]
[UPLOADED] ../dataset/tmp/front_1777416343537_대구06라5245.BMP (Wa1yjTKW0oJkYdJ7JPY4) [7.6s]
[UPLOADED] ../dataset/tmp/front_1777419410056_006마8899.BMP (QQiVOKq92XAR2VwKmR8E) [7.7s]
[UPLOADED] ..

In [5]:
dataset = project.version(1).download("yolo26")

Exporting format yolo26 in progress : 95.0%
Version export complete for yolo26 format



Extracting Dataset Version Zip to quarry-x-object-detection-1 in yolo26:: 100%|██████████| 197/197 [00:00<00:00, 1130.83it/s]


In [6]:
dataset.location

'c:\\Users\\seung\\WAFF\\2026_업체\\대서산업\\ai_vision\\anpr_poc\\quarry-X-Vision-AI\\scripts\\quarry-x-object-detection-1'

In [15]:
import os
import re
import shutil

# 1. 경로 설정
SUB_SETS = ['train', 'valid', 'test']

def get_original_info(rf_filename):
    """
    Roboflow 파일명에서 원본 파일명과 확장자를 복원합니다.
    """
    # .rf.해시값... 제거
    clean = re.sub(r'\.rf\.[a-z0-9]+', '', rf_filename, flags=re.IGNORECASE)
    
    # 확장자 복원 및 원본 확장자 식별
    actual_name = clean
    if '_BMP' in clean:
        actual_name = clean.replace('_BMP', '.BMP')
    elif '_jpg' in clean:
        actual_name = clean.replace('_jpg', '.jpg')
    
    # 확장자를 제외한 순수 파일명 (라벨 매칭용)
    pure_name = os.path.splitext(actual_name)[0]

    part = pure_name.split('_')

    return actual_name, pure_name, part

def find_files_by_parts(target_folder, search_0, search_1):
    """
    target_folder: 검색할 폴더 경로
    search_0: 0번째 인덱스에서 찾을 값 (예: 'front')
    search_1: 1번째 인덱스에서 찾을 값 (예: '1777416343537')
    """
    matched_files = []

    # 1. 폴더 내 파일 목록 가져오기
    for filename in os.listdir(target_folder):
        # 2. Roboflow 특유의 해시값 제거 로직 적용
        # .rf. 및 뒤의 해시 문자열 삭제
        clean = re.sub(r'\.rf\.[a-z0-9]+', '', filename, flags=re.IGNORECASE)
        
        # 3. 확장자 제거 후 순수 파일명만 추출
        pure_name = os.path.splitext(clean)[0]
        
        # 4. 언더바(_) 기준으로 분리
        parts = pure_name.split('_')
        
        # 5. 인덱스 0과 1이 존재하는지 확인 후 매칭 여부 체크
        if len(parts) >= 2:
            if parts[0] == search_0 and parts[1] == search_1:
                # 일치하는 경우 전체 경로를 리스트에 추가
                matched_files.append(filename)
                
    return matched_files

def process_replacement(base_path):
    for subset in SUB_SETS:
        subset_path = os.path.join(base_path, subset)
        if not os.path.exists(subset_path): continue
            
        print(f"\n--- [{subset.upper()}] 원본 교체 및 라벨 정리 시작 ---")
        
        img_dir = os.path.join(subset_path, 'images')
        lbl_dir = os.path.join(subset_path, 'labels')
        
        if not os.path.exists(img_dir): continue

        img_files = os.listdir(img_dir)
        success_count = 0

        for f in img_files:
            old_img_path = os.path.join(img_dir, f)
            if not os.path.isfile(old_img_path): continue

            # 1. 정보 추출
            original_filename, pure_name, part = get_original_info(f)
            origin_file = find_files_by_parts(src_folder, part[0], part[1])
            source_file_path = os.path.join(src_folder, origin_file[0])
            print('source_file_path: ',source_file_path)

            # 2. 이미지 교체 (Source -> Destination)
            if os.path.exists(source_file_path):
                # 기존 Roboflow 이미지 삭제
                os.remove(old_img_path)
                # 원본 소스에서 새 이미지 복사
                shutil.copy2(source_file_path, os.path.join(img_dir, original_filename))
                
                # 3. 라벨 파일 이름 변경 (이미지 이름과 일치시키기)
                # 기존 라벨 파일 찾기 (파일명에 해시가 포함되어 있으므로 glob 패턴처럼 검색)
                rf_label_name = f.replace(os.path.splitext(f)[1], '.txt')
                old_lbl_path = os.path.join(lbl_dir, rf_label_name)
                new_lbl_path = os.path.join(lbl_dir, pure_name + '.txt')

                if os.path.exists(old_lbl_path):
                    if old_lbl_path != new_lbl_path:
                        if os.path.exists(new_lbl_path): os.remove(new_lbl_path)
                        os.rename(old_lbl_path, new_lbl_path)
                
                success_count += 1
            else:
                print(f"  ⚠️ 원본 소스 없음: {source_file_path}")

        print(f"  ✅ {subset} 완료: {success_count}개 파일 교체됨")

if __name__ == "__main__":
    process_replacement(dataset.location)
    print(dataset.location)
    print("\n✨ 모든 이미지가 원본 소스로 대체되었고 라벨 이름이 동기화되었습니다!")


--- [TRAIN] 원본 교체 및 라벨 정리 시작 ---
source_file_path:  ..\dataset\20260429\front_1777416275099_006마8899.BMP
source_file_path:  ..\dataset\20260429\front_1777416343537_대구06라5245.BMP
source_file_path:  ..\dataset\20260429\front_1777416835603_대구06라5245.jpg
source_file_path:  ..\dataset\20260429\front_1777419410056_006마8899.BMP
source_file_path:  ..\dataset\20260429\front_1777419540423_경북06모6547.BMP
source_file_path:  ..\dataset\20260429\front_1777419570815_대구06라5245.BMP
source_file_path:  ..\dataset\20260429\front_1777419725549_006마8899.jpg
source_file_path:  ..\dataset\20260429\front_1777420068752_경북06모6547.jpg
source_file_path:  ..\dataset\20260429\front_1777420716128_006나7520.BMP
source_file_path:  ..\dataset\20260429\front_1777420827236_006다6192.BMP
source_file_path:  ..\dataset\20260429\front_1777420957018_006나7520.jpg
source_file_path:  ..\dataset\20260429\front_1777421066083_대구06라5226.jpg
source_file_path:  ..\dataset\20260429\front_1777421980474_대구06라5245.BMP
source_file_path:  ..\d

In [20]:
import os
from ftplib import FTP
from tqdm import tqdm

def ftp_makedirs(ftp, remote_path):
    """서버에 중첩된 폴더 구조를 자동으로 생성하는 함수"""
    path_parts = [p for p in remote_path.split('/') if p]
    current_path = ""
    for part in path_parts:
        current_path += f"/{part}"
        try:
            ftp.cwd(current_path)
        except:
            ftp.mkd(current_path)
            ftp.cwd(current_path)

def ftp_rmtree(ftp, path):
    """업로드 전에 기존 폴더 삭제"""
    try:
        ftp.cwd(path)
    except Exception:
        return  # 폴더 없음

    file_list = ftp.nlst()

    for name in file_list:
        full_path = f"{path}/{name}"
        try:
            ftp.cwd(full_path)
            ftp.cwd("..")
            ftp_rmtree(ftp, full_path)
        except Exception:
            ftp.delete(full_path)

    ftp.rmd(path)

# 1. 설정
FTP_HOST = "192.168.1.179"
FTP_USER = "administrator"
FTP_PASS = "waff!23"
REMOTE_FOLDER = "/ai_vision/yolo"
SUB_SETS = ['train', 'valid', 'test']

# 2. FTP 연결
ftp = FTP()
ftp.connect(FTP_HOST, 21)
ftp.login(user=FTP_USER, passwd=FTP_PASS)
ftp.encoding = "utf-8"  # 한글 깨짐 방지

try:
    print("기존 subset 폴더 삭제 중...")

    for subset in SUB_SETS:
        remote_subset_path = f"{REMOTE_FOLDER}/{subset}"
        ftp_rmtree(ftp, remote_subset_path)

    print("subset 폴더 삭제 완료")

    # 모든 파일 목록 먼저 수집 (tqdm 전체 개수 파악용)
    all_files = []
    for root, dirs, files in os.walk(dataset.location):
        for file in files:
            all_files.append(os.path.join(root, file))

    print(f"총 {len(all_files)}개의 파일을 업로드합니다.")

    # 3. 폴더 순회 및 업로드
    for local_file in tqdm(all_files, desc="업로드 및 로컬 삭제 중", unit="file"):
        # 로컬 상대 경로를 추출하여 서버 경로 생성
        rel_path = os.path.relpath(local_file, dataset.location)
        remote_file_path = os.path.join(REMOTE_FOLDER, rel_path).replace("\\", "/")
        remote_dir = os.path.dirname(remote_file_path)

        # 서버에 해당 폴더가 있는지 확인 및 생성
        ftp_makedirs(ftp, remote_dir)

        # 파일 전송
        try:
            with open(local_file, "rb") as f:
                # 파일 전송 명령
                ftp.storbinary(f"STOR {os.path.basename(remote_file_path)}", f)
            
            # storbinary가 성공적으로 끝나면 파일 삭제 실행
            os.remove(local_file) 
            
        except Exception as file_error:
            print(f"\n파일 처리 오류 ({local_file}): {file_error}")
            # 특정 파일 업로드 실패 시 삭제하지 않고 다음 파일로 진행

    print("\n--- 폴더 전체 복사 완료 ---")

except Exception as e:
    # 에러 메시지에 '226'이나 'Transfer complete'가 있다면 이는 성공으로 간주
    if "226" in str(e) or "success" in str(e).lower():
        print("\n[알림] 서버의 성공 응답을 확인했습니다.")
    else:
        print(f"\n[실제 오류 발생]: {e}")

finally:
    # 안전하게 닫기
    try:
        ftp.quit()
    except:
        ftp.close()

# 2. 환영 메시지 및 현재 파일 목록 출력
#print(ftp.getwelcome())
#ftp.dir() 

# 2. 파일 목록 조회
# nlst(): 파일/폴더 이름만 리스트로 반환 (프로그래밍하기 편함)
#files = ftp.nlst()

기존 subset 폴더 삭제 중...
subset 폴더 삭제 완료
총 195개의 파일을 업로드합니다.


업로드 및 로컬 삭제 중: 100%|██████████| 195/195 [00:18<00:00, 10.70file/s]


--- 폴더 전체 복사 완료 ---


In [70]:
import paramiko

# --- 서버 접속 정보 ---
HOST = "192.168.1.179"
USER = "administrator"
PASS = "waff!23"

# 서버에 이미 존재하는 파일의 전체 경로
# 1. 가상환경 내 파이썬 경로
VENV_PYTHON = r'D:\ai_vision\yolo\venv\Scripts\python.exe'
# 2. 이동할 작업 디렉토리
WORKING_DIR = r'D:\ai_vision\yolo'
# 3. 실행할 스크립트 파일 이름 (이미 해당 폴더 안이라면 파일명만 써도 됨)
SCRIPT_NAME = 'train.py'
def test_ssh_execution():
    print(f"[{HOST}] 연결 시도 중...")
    
    ssh = paramiko.SSHClient()
    # 서버의 호스트 키가 없어도 자동으로 등록하도록 설정
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    
    try:
        # 1. SSH 접속
        ssh.connect(HOST, username=USER, password=PASS)
        print("✅ SSH 접속 성공!")

        # 2. 명령어 실행 (PowerShell을 통해 가상환경 실행)
        command = f'powershell.exe -Command "Set-Location \'{WORKING_DIR}\'; & \'{VENV_PYTHON}\' \'{SCRIPT_NAME}\'"'
        print(f"실행 명령어: {command}")
        # 2. 명령어 실행 (PowerShell을 통해 Python 실행)
        #command = f'powershell.exe -Command "python {REMOTE_FILE_PATH}"'
        #print(f"실행 명령어: {command}")
        
        stdin, stdout, stderr = ssh.exec_command(command)

        # 3. 결과 읽기 (윈도우 서버 결과는 cp949 인코딩이 많음)
        output = stdout.read().decode('cp949', errors='ignore')
        error = stderr.read().decode('cp949', errors='ignore')

        if output:
            print("\n--- [실행 결과] ---")
            print(output)
        
        if error:
            print("\n--- [에러 메시지] ---")
            print(error)
            print("💡 팁: 서버에 Python이 설치되어 있고 환경변수(Path)에 등록되어 있는지 확인하세요.")

    except Exception as e:
        print(f"❌ 접속 또는 실행 중 오류 발생: {e}")
        
    finally:
        ssh.close()
        print("\nSSH 연결 종료.")

if __name__ == "__main__":
    test_ssh_execution()

[192.168.1.179] 연결 시도 중...
✅ SSH 접속 성공!
실행 명령어: powershell.exe -Command "Set-Location 'D:\ai_vision\yolo'; & 'D:\ai_vision\yolo\venv\Scripts\python.exe' 'train.py'"

SSH 연결 종료.


KeyboardInterrupt: 